In [5]:
# ============================================================
# Cell 1 — VATN LOSO: Environment and Configuration
# ============================================================

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gc
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    TimeDistributed,
    Flatten,
    Conv1D,
    MultiHeadAttention,
    Add,
    LayerNormalization,
    GlobalAveragePooling1D,
    Dense,
    Activation,
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# VATN input settings
IMG_SIZE = 64
MAX_FRAMES = 30
BATCH_SIZE = 8
EPOCHS = 60
PATIENCE = 10

# Dataset and result paths
ROOT = Path(r"D:\LOSO_MEDIAPIPE")
METADATA_PATH = ROOT / "metadata.csv"
RESULTS_DIR = ROOT / "vatn_loso_models"

assert METADATA_PATH.exists(), f"metadata.csv not found: {METADATA_PATH}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# GPU Setup
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Using GPU: {gpus[0]}")
    except RuntimeError as e:
        print(e)

# GPU memory configuration
gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as error:
        print("GPU memory-growth warning:", error)

print("=" * 70)
print("VATN LOSO environment initialized")
print("=" * 70)
print("TensorFlow version :", tf.__version__)
print("NumPy version      :", np.__version__)
print("OpenCV version     :", cv2.__version__)
print("GPU devices        :", gpus)
print("MAX_FRAMES         :", MAX_FRAMES)
print("Image size         :", f"{IMG_SIZE} x {IMG_SIZE}")
print("Metadata path      :", METADATA_PATH)
print("Results directory  :", RESULTS_DIR)

VATN LOSO environment initialized
TensorFlow version : 2.19.0
NumPy version      : 1.26.4
OpenCV version     : 4.10.0
GPU devices        : []
MAX_FRAMES         : 30
Image size         : 64 x 64
Metadata path      : D:\LOSO_MEDIAPIPE\metadata.csv
Results directory  : D:\LOSO_MEDIAPIPE\vatn_loso_models


In [6]:
# ============================================================
# Cell 2 — Load Metadata and Create LOSO Folds
# ============================================================

metadata = pd.read_csv(METADATA_PATH)

required_columns = {
    "feature_file",
    "video_path",
    "signer",
    "class_name",
    "label",
}

missing_columns = required_columns - set(metadata.columns)
assert not missing_columns, (
    f"metadata.csv is missing columns: {missing_columns}"
)

# Verify that the original RGB videos required by VATN exist.
metadata["video_exists"] = metadata["video_path"].map(
    lambda path: Path(path).is_file()
)

missing_videos = metadata.loc[
    ~metadata["video_exists"],
    ["video_path", "signer", "class_name"]
]

if not missing_videos.empty:
    print("Missing source videos:", len(missing_videos))
    display(missing_videos.head(10))
    raise FileNotFoundError(
        "VATN requires the original RGB videos. "
        "Correct video_path entries in metadata.csv first."
    )

# Create stable labels from sorted class names.
class_names = sorted(metadata["class_name"].unique())
class_to_label = {
    class_name: label
    for label, class_name in enumerate(class_names)
}

metadata["label_index"] = metadata["class_name"].map(
    class_to_label
).astype(np.int32)

video_paths = metadata["video_path"].to_numpy()
labels = metadata["label_index"].to_numpy(dtype=np.int32)
groups = metadata["signer"].to_numpy()

NUM_CLASSES = len(class_names)

# Create true Leave-One-Signer-Out folds.
logo = LeaveOneGroupOut()
folds = list(logo.split(video_paths, labels, groups))

print("=" * 70)
print("Metadata loaded successfully")
print("=" * 70)
print("Total videos :", len(metadata))
print("Total classes:", NUM_CLASSES)
print("Total signers:", len(np.unique(groups)))
print("Classes      :", class_names[:5], "...", class_names[-5:])
print("Signers      :", sorted(np.unique(groups)))

print("\nLOSO folds:")
for fold_number, (train_idx, test_idx) in enumerate(folds, start=1):
    print(
        f"Fold {fold_number}: "
        f"Train = {list(np.unique(groups[train_idx]))}, "
        f"Test = {list(np.unique(groups[test_idx]))}, "
        f"Train samples = {len(train_idx)}, "
        f"Test samples = {len(test_idx)}"
    )

Metadata loaded successfully
Total videos : 3200
Total classes: 50
Total signers: 4
Classes      : ['BLAZER', 'BRUSHING', 'Document folder', 'Family', 'Father'] ... ['visit', 'walking', 'wife', 'work', 'writing']
Signers      : ['S1', 'S2', 'S3', 'S4']

LOSO folds:
Fold 1: Train = ['S2', 'S3', 'S4'], Test = ['S1'], Train samples = 2400, Test samples = 800
Fold 2: Train = ['S1', 'S3', 'S4'], Test = ['S2'], Train samples = 2400, Test samples = 800
Fold 3: Train = ['S1', 'S2', 'S4'], Test = ['S3'], Train samples = 2400, Test samples = 800
Fold 4: Train = ['S1', 'S2', 'S3'], Test = ['S4'], Train samples = 2400, Test samples = 800


In [7]:
# ============================================================
# Cell 3 — VATN RGB Video Loader and Frame-Sampling Test
# ============================================================

def load_video_frames(video_path, max_frames=MAX_FRAMES):
    """
    Load exactly max_frames uniformly sampled RGB frames.

    Output shape: (max_frames, IMG_SIZE, IMG_SIZE, 3)
    Output dtype: float32, normalized to [0, 1].
    """
    video_path = str(video_path)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None, "open_failed"

    try:
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            return None, "empty_video"

        frame_indices = np.linspace(
            0,
            total_frames - 1,
            num=max_frames
        ).round().astype(int)

        sampled_frames = []
        last_valid_frame = None

        for frame_index in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            success, frame = cap.read()

            if success:
                frame = cv2.resize(
                    frame,
                    (IMG_SIZE, IMG_SIZE),
                    interpolation=cv2.INTER_AREA
                )

                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                last_valid_frame = frame

            # Repeat the previous valid frame if decoding fails.
            if last_valid_frame is None:
                sampled_frames.append(
                    np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
                )
            else:
                sampled_frames.append(last_valid_frame.copy())

        video = np.asarray(sampled_frames, dtype=np.float32) / 255.0

        assert video.shape == (
            max_frames,
            IMG_SIZE,
            IMG_SIZE,
            3
        ), video.shape

        return video, "ok"

    finally:
        cap.release()


# Test one video before processing a full LOSO fold.
test_video_path = video_paths[0]

test_video, test_status = load_video_frames(test_video_path)

print("=" * 70)
print("VATN video-loader test")
print("=" * 70)
print("Video path :", test_video_path)
print("Status     :", test_status)

if test_video is not None:
    print("Shape      :", test_video.shape)
    print("Dtype      :", test_video.dtype)
    print("Min value  :", test_video.min())
    print("Max value  :", test_video.max())
    print("Frames     :", test_video.shape[0])

VATN video-loader test
Video path : D:\LOSO DATASET\S1\BLAZER\IMG_5590_bright.mp4
Status     : ok
Shape      : (30, 64, 64, 3)
Dtype      : float32
Min value  : 0.0
Max value  : 0.99607843
Frames     : 30


In [8]:
# ============================================================
# Cell 4 — TensorFlow VATN Video Dataset Pipeline
# ============================================================

def load_video_for_tensorflow(path_value):
    """Wrapper used inside tf.data; always returns a valid video tensor."""
    if isinstance(path_value, np.ndarray):
        path_value = path_value.item()

    if isinstance(path_value, bytes):
        path_value = path_value.decode("utf-8")

    video, status = load_video_frames(str(path_value))

    if status != "ok" or video is None:
        return np.zeros(
            (MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3),
            dtype=np.float32
        )

    return video.astype(np.float32)


def decode_video_tf(path, label):
    video = tf.numpy_function(
        func=load_video_for_tensorflow,
        inp=[path],
        Tout=tf.float32
    )

    video.set_shape((MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3))
    label.set_shape(())

    return video, label


@tf.function
def augment_video_tf(video, label):
    """Training-only VATN RGB-video augmentation."""
    video = tf.image.random_brightness(video, max_delta=0.04)
    video = tf.image.random_contrast(video, lower=0.95, upper=1.05)

    # Small spatial translation/crop, applied consistently to the full sequence.
    padded = tf.image.resize_with_crop_or_pad(
        video,
        IMG_SIZE + 2,
        IMG_SIZE + 2
    )

    video = tf.image.random_crop(
        padded,
        size=(MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3)
    )

    noise = tf.random.normal(
        tf.shape(video),
        mean=0.0,
        stddev=0.008,
        dtype=tf.float32
    )

    video = tf.clip_by_value(video + noise, 0.0, 1.0)

    return video, label


def create_vatn_dataset(paths, target_labels, training=False):
    """
    Create a tf.data pipeline.
    Videos are decoded only when required, avoiding loading all
    3,200 RGB video sequences into RAM.
    """
    dataset = tf.data.Dataset.from_tensor_slices((
        np.asarray(paths, dtype=str),
        np.asarray(target_labels, dtype=np.int32)
    ))

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(paths), 2048),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        decode_video_tf,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        dataset = dataset.map(
            augment_video_tf,
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


# Test one evaluation batch before starting LOSO training.
pipeline_test_ds = create_vatn_dataset(
    video_paths[:BATCH_SIZE],
    labels[:BATCH_SIZE],
    training=False
)

batch_videos, batch_labels = next(iter(pipeline_test_ds))

print("=" * 70)
print("VATN tf.data pipeline test")
print("=" * 70)
print("Batch video shape:", batch_videos.shape)
print("Batch video dtype:", batch_videos.dtype)
print("Batch labels:", batch_labels.numpy())
print("Batch minimum:", float(tf.reduce_min(batch_videos)))
print("Batch maximum:", float(tf.reduce_max(batch_videos)))

VATN tf.data pipeline test
Batch video shape: (8, 30, 64, 64, 3)
Batch video dtype: <dtype: 'float32'>
Batch labels: [0 0 0 0 0 0 0 0]
Batch minimum: 0.0
Batch maximum: 0.9960784316062927


In [9]:
# ============================================================
# Cell 5 — VATN Model Architecture
# ============================================================

def build_vatn_model(num_classes):
    """
    Hybrid VATN:
    TimeDistributed CNN → Temporal Conv1D → Self-Attention → Classifier.
    """

    inputs = Input(
        shape=(MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3),
        name="rgb_video"
    )

    # Spatial feature extraction for each RGB frame.
    x = TimeDistributed(
        Conv2D(32, (3, 3), padding="same", activation="relu"),
        name="block1_conv"
    )(inputs)

    x = TimeDistributed(
        BatchNormalization(),
        name="block1_bn"
    )(x)

    x = TimeDistributed(
        MaxPooling2D((2, 2)),
        name="block1_pool"
    )(x)

    x = TimeDistributed(
        Dropout(0.15),
        name="block1_dropout"
    )(x)

    x = TimeDistributed(
        Conv2D(64, (3, 3), padding="same", activation="relu"),
        name="block2_conv"
    )(x)

    x = TimeDistributed(
        BatchNormalization(),
        name="block2_bn"
    )(x)

    x = TimeDistributed(
        MaxPooling2D((2, 2)),
        name="block2_pool"
    )(x)

    x = TimeDistributed(
        Dropout(0.15),
        name="block2_dropout"
    )(x)

    x = TimeDistributed(
        Conv2D(128, (3, 3), padding="same", activation="relu"),
        name="block3_conv"
    )(x)

    x = TimeDistributed(
        BatchNormalization(),
        name="block3_bn"
    )(x)

    x = TimeDistributed(
        MaxPooling2D((2, 2)),
        name="block3_pool"
    )(x)

    x = TimeDistributed(
        Dropout(0.20),
        name="block3_dropout"
    )(x)

    # Convert each frame feature map into one temporal feature vector.
    x = TimeDistributed(
        Flatten(),
        name="flatten"
    )(x)

    # Temporal convolution network.
    x = Conv1D(
        128,
        kernel_size=3,
        padding="causal",
        activation="relu",
        name="tcn1"
    )(x)

    x = BatchNormalization(name="tcn1_bn")(x)
    x = Dropout(0.20, name="tcn1_dropout")(x)

    x = Conv1D(
        128,
        kernel_size=3,
        padding="causal",
        activation="relu",
        name="tcn2"
    )(x)

    x = BatchNormalization(name="tcn2_bn")(x)
    x = Dropout(0.20, name="tcn2_dropout")(x)

    # Temporal self-attention.
    attention = MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        dropout=0.15,
        name="temporal_attention"
    )(x, x)

    x = Add(name="attention_residual")([x, attention])
    x = LayerNormalization(
        epsilon=1e-6,
        name="attention_norm"
    )(x)

    # Sequence-level classification.
    x = GlobalAveragePooling1D(name="global_temporal_pool")(x)

    x = Dense(
        256,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
        name="dense1"
    )(x)

    x = Dropout(0.30, name="dense1_dropout")(x)

    outputs = Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="class_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="VATN_LOSO"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=3e-4,
            clipnorm=1.0
        ),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=3,
                name="top_3_accuracy"
            )
        ]
    )

    return model


# Build once only to verify the architecture.
tf.keras.backend.clear_session()

vatn_test_model = build_vatn_model(NUM_CLASSES)

print("=" * 70)
print("VATN model architecture")
print("=" * 70)

vatn_test_model.summary()


VATN model architecture


Model: "VATN_LOSO"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ rgb_video (InputLayer)        │ (None, 30, 64, 64, 3)     │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv (TimeDistributed) │ (None, 30, 64, 64, 32)    │             896 │ rgb_video[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_bn (TimeDistributed)   │ (None, 30, 64, 64, 32)    │             128 │ block1_conv[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_pool (TimeDistributed) │ (None, 30, 32, 32, 32)    │               0 │ block1_bn[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_dropout                │ (None, 30, 32, 32, 32)    │               0 │ block1_pool[0][0]          │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_conv (TimeDistributed) │ (None, 30, 32, 32, 64)    │          18,496 │ block1_dropout[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_bn (TimeDistributed)   │ (None, 30, 32, 32, 64)    │             256 │ block2_conv[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_pool (TimeDistributed) │ (None, 30, 16, 16, 64)    │               0 │ block2_bn[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_dropout                │ (None, 30, 16, 16, 64)    │               0 │ block2_pool[0][0]          │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block3_conv (TimeDistributed) │ (None, 30, 16, 16, 128)   │          73,856 │ block2_dropout[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block3_bn (TimeDistributed)   │ (None, 30, 16, 16, 128)   │             512 │ block3_conv[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block3_pool (TimeDistributed) │ (None, 30, 8, 8, 128)     │               0 │ block3_bn[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block3_dropout                │ (None, 30, 8, 8, 128)     │               0 │ block3_pool[0][0]          │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (TimeDistributed)     │ (None, 30, 8192)          │               0 │ block3_dropout[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ tcn1 (Conv1D)                 │ (None, 30, 128)           │       3,145,856 │ flatten[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ tcn1_bn (BatchNormalization)  │ (None, 30, 128)           │             51

 Total params: 3,402,482 (12.98 MB)

 Trainable params: 3,401,522 (12.98 MB)

 Non-trainable params: 960 (3.75 KB)

In [10]:
# ============================================================
# Cell 6 — Prepare LOSO Fold 1 Datasets
# ============================================================

CURRENT_FOLD = 1

train_indices, test_indices = folds[CURRENT_FOLD - 1]

test_signer = np.unique(groups[test_indices])[0]

train_paths_full = video_paths[train_indices]
train_labels_full = labels[train_indices]

test_paths = video_paths[test_indices]
test_labels = labels[test_indices]

# Validation videos are selected only from the three training signers.
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths_full,
    train_labels_full,
    test_size=0.15,
    random_state=SEED + CURRENT_FOLD,
    stratify=train_labels_full
)

# Build memory-efficient video datasets.
train_ds = create_vatn_dataset(
    train_paths,
    train_labels,
    training=True
)

val_ds = create_vatn_dataset(
    val_paths,
    val_labels,
    training=False
)

test_ds = create_vatn_dataset(
    test_paths,
    test_labels,
    training=False
)

print("=" * 70)
print(f"VATN LOSO Fold {CURRENT_FOLD} prepared")
print("=" * 70)
print("Test signer       :", test_signer)
print("Training samples  :", len(train_paths))
print("Validation samples:", len(val_paths))
print("Testing samples   :", len(test_paths))
print("Training signers  :", sorted(np.unique(groups[train_indices])))
print("Testing signer    :", test_signer)

# Verify one training batch.
fold_batch_videos, fold_batch_labels = next(iter(train_ds))

print("\nTraining batch shape :", fold_batch_videos.shape)
print("Training batch labels:", fold_batch_labels.numpy())

VATN LOSO Fold 1 prepared
Test signer       : S1
Training samples  : 2040
Validation samples: 360
Testing samples   : 800
Training signers  : ['S2', 'S3', 'S4']
Testing signer    : S1

Training batch shape : (8, 30, 64, 64, 3)
Training batch labels: [36 40 29  6 11 10 44 47]


In [ ]:
# ============================================================
# Cell 7 — Train VATN on LOSO Fold 1
# ============================================================

# Remove the architecture-test model before starting actual training.
del vatn_test_model
gc.collect()
tf.keras.backend.clear_session()

# Create a new model for Fold 1 only.
model = build_vatn_model(NUM_CLASSES)

fold_model_path = RESULTS_DIR / (
    f"vatn_loso_fold_{CURRENT_FOLD}_test_{test_signer}.keras"
)

callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=str(fold_model_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

print("=" * 70)
print(f"Training VATN LOSO Fold {CURRENT_FOLD}")
print(f"Test signer: {test_signer}")
print(f"Model output: {fold_model_path}")
print("=" * 70)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

Training VATN LOSO Fold 1
Test signer: S1
Model output: D:\LOSO_MEDIAPIPE\vatn_loso_models\vatn_loso_fold_1_test_S1.keras
Epoch 1/60
111/255 ━━━━━━━━━━━━━━━━━━━━ 2:07:53 53s/step - accuracy: 0.0320 - loss: 4.0180 - top_3_accuracy: 0.0922